In [1]:

"""
CN7 08 - Logistic 정밀 튜닝 + OOF Blend Screening
==================================================

CN7_07 보완점
-------------
CN7_06 baseline Logistic:
    class_weight="balanced"

CN7_07:
    fixed positive weight 20 / 70만 비교

따라서 CN7_08에서는 다음을 동일 검증 구조에서 다시 비교한다.

1) Nested train-only Top-K selection
2) class_weight:
   - balanced
   - fixed 20
   - fixed 70
3) C:
   - 0.1
   - 0.3
4) scaler:
   - Standard
   - Robust
5) Side:
   - on / off
6) Top-K:
   - 10 / 15 / 23

이후 개발용 OOF ensemble screening:
- 상위 single model 후보들을 추림
- probability mean
- rank mean
을 비교

주의
----
Ensemble 후보 선택과 평가가 같은 OOF를 사용하므로
앙상블 점수는 "개발용 후보 선정"에만 사용한다.
최종 독립 성능 추정치라고 부르지 않는다.
"""

from pathlib import Path
from itertools import product, combinations
import warnings
import numpy as np
import pandas as pd

from scipy.stats import ks_2samp

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

# =============================================================================
# 0. 설정
# =============================================================================

CN7_WORK_DIR = Path(
    r"C:\Users\Administrator\Desktop\kamp\다시하는 cn7"
)

RAW_PATH = (
    CN7_WORK_DIR
    / "cn7_03_eda"
    / "cn7_train_raw_1211.csv"
)

OUT_DIR = (
    CN7_WORK_DIR
    / "cn7_08_refine_ensemble"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "PassOrFail"

SENSOR_COLS = [
    "Injection_Time",
    "Filling_Time",
    "Plasticizing_Time",
    "Cycle_Time",
    "Clamp_Close_Time",
    "Cushion_Position",
    "Plasticizing_Position",
    "Clamp_Open_Position",
    "Max_Injection_Speed",
    "Max_Screw_RPM",
    "Average_Screw_RPM",
    "Max_Injection_Pressure",
    "Max_Switch_Over_Pressure",
    "Max_Back_Pressure",
    "Average_Back_Pressure",
    "Barrel_Temperature_1",
    "Barrel_Temperature_2",
    "Barrel_Temperature_3",
    "Barrel_Temperature_4",
    "Barrel_Temperature_5",
    "Barrel_Temperature_6",
    "Hopper_Temperature",
    "Mold_Temperature_3",
    "Mold_Temperature_4",
]

TOP_K_VALUES = [10, 15, 23]
SCALERS = ["standard", "robust"]
C_VALUES = [0.1, 0.3]
WEIGHT_MODES = ["balanced", "fixed20", "fixed70"]
USE_SIDE_VALUES = [False, True]

GROUP_SEEDS = [42, 52, 62, 72, 82]
N_SPLITS = 5

TIME_BLOCK_MINUTES = 10
PURGE_MINUTES = 10

TOP_POOL_PER_CRITERION = 4


# =============================================================================
# 유틸
# =============================================================================

def banner(title):
    print("\n" + "=" * 120)
    print(title)
    print("=" * 120)


def make_config_id(top_k, scaler, c_value, weight_mode, use_side):
    return (
        f"k{top_k}"
        f"__{scaler}"
        f"__C{c_value:g}"
        f"__{weight_mode}"
        f"__side{int(use_side)}"
    )


def effect_shift_score(normal, defect):
    normal = np.asarray(normal, dtype=float)
    defect = np.asarray(defect, dtype=float)

    if len(normal) == 0 or len(defect) == 0:
        return 0.0

    sd = np.nanstd(normal, ddof=0)

    if not np.isfinite(sd) or sd < 1e-12:
        return 0.0

    return float(
        abs(np.nanmean(defect) - np.nanmean(normal)) / sd
    )


def rank_features_train(df_train, feature_cols, target_col):
    y = df_train[target_col].astype(int)

    rows = []

    for c in feature_cols:
        x0 = pd.to_numeric(
            df_train.loc[y == 0, c],
            errors="coerce",
        ).dropna()

        x1 = pd.to_numeric(
            df_train.loc[y == 1, c],
            errors="coerce",
        ).dropna()

        if len(x0) == 0 or len(x1) == 0:
            ks_stat = 0.0
            shift = 0.0
        else:
            try:
                ks_stat = float(
                    ks_2samp(x0, x1).statistic
                )
            except Exception:
                ks_stat = 0.0

            shift = effect_shift_score(
                x0,
                x1,
            )

        score = (
            ks_stat
            + 0.5 * min(shift, 4.0)
        )

        rows.append({
            "feature": c,
            "KS_stat": ks_stat,
            "mean_shift_normal_sd": shift,
            "selection_score": score,
        })

    result = (
        pd.DataFrame(rows)
        .sort_values(
            [
                "selection_score",
                "KS_stat",
                "feature",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    result["rank"] = (
        np.arange(len(result)) + 1
    )

    return result


def resolve_class_weight(weight_mode):
    if weight_mode == "balanced":
        return "balanced"

    if weight_mode == "fixed20":
        return {
            0: 1.0,
            1: 20.0,
        }

    if weight_mode == "fixed70":
        return {
            0: 1.0,
            1: 70.0,
        }

    raise ValueError(weight_mode)


def make_model(
    scaler_name,
    c_value,
    weight_mode,
    random_state,
):
    if scaler_name == "standard":
        scaler = StandardScaler()
    elif scaler_name == "robust":
        scaler = RobustScaler()
    else:
        raise ValueError(scaler_name)

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
            ),
        ),
        (
            "scaler",
            scaler,
        ),
        (
            "model",
            LogisticRegression(
                C=c_value,
                penalty="l2",
                class_weight=resolve_class_weight(
                    weight_mode
                ),
                max_iter=5000,
                solver="liblinear",
                random_state=random_state,
            ),
        ),
    ])


def score_predictions(y_true, score):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(score).astype(float)

    valid = np.isfinite(s)
    y = y[valid]
    s = s[valid]

    if len(y) == 0 or len(np.unique(y)) < 2:
        return {
            "n": len(y),
            "defects": int(y.sum()),
            "PR_AUC": np.nan,
            "ROC_AUC": np.nan,
        }

    return {
        "n": len(y),
        "defects": int(y.sum()),
        "PR_AUC": float(
            average_precision_score(y, s)
        ),
        "ROC_AUC": float(
            roc_auc_score(y, s)
        ),
    }


def threshold_table(y_true, score):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(score).astype(float)

    valid = np.isfinite(s)
    y = y[valid]
    s = s[valid]

    thresholds = np.sort(
        np.unique(s)
    )[::-1]

    rows = []

    for threshold in thresholds:
        pred = (
            s >= threshold
        ).astype(int)

        tn, fp, fn, tp = (
            confusion_matrix(
                y,
                pred,
                labels=[0, 1],
            ).ravel()
        )

        precision = (
            tp / (tp + fp)
            if tp + fp > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if tp + fn > 0
            else 0.0
        )

        rows.append({
            "threshold": float(threshold),
            "precision": float(precision),
            "recall": float(recall),
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn),
            "predicted_positive": int(tp + fp),
        })

    return pd.DataFrame(rows)


def best_precision_at_recall(
    y_true,
    score,
    recall_target,
):
    tab = threshold_table(
        y_true,
        score,
    )

    cand = tab[
        tab["recall"] >= recall_target
    ].copy()

    if len(cand) == 0:
        return {
            "precision": np.nan,
            "recall": np.nan,
            "threshold": np.nan,
            "TP": np.nan,
            "FP": np.nan,
            "FN": np.nan,
        }

    cand = cand.sort_values(
        [
            "precision",
            "recall",
            "predicted_positive",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )

    r = cand.iloc[0]

    return {
        "precision": float(r["precision"]),
        "recall": float(r["recall"]),
        "threshold": float(r["threshold"]),
        "TP": int(r["TP"]),
        "FP": int(r["FP"]),
        "FN": int(r["FN"]),
    }


def top_fraction_metric(
    y_true,
    score,
    frac=0.05,
):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(score).astype(float)

    valid = np.isfinite(s)
    y = y[valid]
    s = s[valid]

    k = max(
        1,
        int(np.ceil(len(y) * frac)),
    )

    order = np.argsort(-s)
    idx = order[:k]

    tp = int(y[idx].sum())
    fp = int(k - tp)
    fn = int(y.sum() - tp)

    return {
        "k": k,
        "precision": float(tp / k),
        "recall": float(
            tp / y.sum()
            if y.sum() > 0
            else np.nan
        ),
        "TP": tp,
        "FP": fp,
        "FN": fn,
    }


def evaluate_score(y, score):
    metric = score_predictions(
        y,
        score,
    )

    r80 = best_precision_at_recall(
        y,
        score,
        0.80,
    )

    r90 = best_precision_at_recall(
        y,
        score,
        0.90,
    )

    top5 = top_fraction_metric(
        y,
        score,
        0.05,
    )

    return {
        "PR_AUC": metric["PR_AUC"],
        "ROC_AUC_ref": metric["ROC_AUC"],

        "R80_precision": r80["precision"],
        "R80_recall": r80["recall"],
        "R80_threshold": r80["threshold"],
        "R80_TP": r80["TP"],
        "R80_FP": r80["FP"],

        "R90_precision": r90["precision"],
        "R90_recall": r90["recall"],
        "R90_threshold": r90["threshold"],
        "R90_TP": r90["TP"],
        "R90_FP": r90["FP"],

        "Top5_precision": top5["precision"],
        "Top5_recall": top5["recall"],
        "Top5_TP": top5["TP"],
        "Top5_FP": top5["FP"],
    }


def percentile_rank_score(score):
    s = pd.Series(
        np.asarray(score, dtype=float)
    )

    return (
        s.rank(
            method="average",
            pct=True,
        )
        .to_numpy()
    )


# =============================================================================
# 1. 데이터
# =============================================================================

banner(
    "CN7 08 - Logistic 정밀 튜닝 + OOF Blend"
)

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"없음: {RAW_PATH}"
    )

df = pd.read_csv(
    RAW_PATH
)

df[TARGET] = pd.to_numeric(
    df[TARGET],
    errors="raise",
).astype(int)

if "TimeStamp_dt" in df.columns:
    df["TimeStamp_dt"] = pd.to_datetime(
        df["TimeStamp_dt"],
        errors="coerce",
    )
else:
    df["TimeStamp_dt"] = pd.to_datetime(
        df["TimeStamp"],
        errors="coerce",
    )

if "Side" not in df.columns:
    s = (
        df["PART_NAME"]
        .astype(str)
        .str.upper()
    )

    df["Side"] = np.where(
        s.str.contains(
            r"\bRH\b",
            regex=True,
            na=False,
        ),
        "RH",
        "LH",
    )

df["Side_RH"] = (
    df["Side"] == "RH"
).astype(int)

constant_cols = [
    c
    for c in SENSOR_COLS
    if (
        c in df.columns
        and df[c].nunique(
            dropna=False
        ) <= 1
    )
]

base_features = [
    c
    for c in SENSOR_COLS
    if (
        c in df.columns
        and c not in constant_cols
    )
]

y = df[TARGET]

print(
    f"shape={df.shape}, "
    f"defects={int(y.sum())}/{len(y)}, "
    f"features={len(base_features)}"
)


# =============================================================================
# 2. Group
# =============================================================================

banner(
    "1) Sensor Group"
)

df["sensor_group_id"] = (
    df.groupby(
        base_features,
        dropna=False,
        sort=False,
    )
    .ngroup()
)

groups = df["sensor_group_id"]

print(
    "groups:",
    groups.nunique(),
)

print(
    "positive groups:",
    int(
        df.groupby(
            "sensor_group_id"
        )[TARGET]
        .max()
        .sum()
    ),
)


# =============================================================================
# 3. configs
# =============================================================================

banner(
    "2) Configs"
)

configs = []

for (
    top_k,
    scaler,
    c_value,
    weight_mode,
    use_side,
) in product(
    TOP_K_VALUES,
    SCALERS,
    C_VALUES,
    WEIGHT_MODES,
    USE_SIDE_VALUES,
):
    configs.append({
        "config_id":
            make_config_id(
                top_k,
                scaler,
                c_value,
                weight_mode,
                use_side,
            ),
        "top_k": top_k,
        "scaler": scaler,
        "C": c_value,
        "weight_mode": weight_mode,
        "use_side": use_side,
    })

print(
    "config count:",
    len(configs),
)


# =============================================================================
# 4. Group folds + rankings
# =============================================================================

banner(
    "3) Group folds"
)

group_folds = []

fold_global_id = 0

for repeat_no, seed in enumerate(
    GROUP_SEEDS,
    start=1,
):
    cv = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed,
    )

    for fold_no, (
        tr_idx,
        va_idx,
    ) in enumerate(
        cv.split(
            df,
            y,
            groups,
        ),
        start=1,
    ):
        fold_global_id += 1

        rank_df = rank_features_train(
            df.iloc[tr_idx],
            base_features,
            TARGET,
        )

        group_folds.append({
            "fold_global_id":
                fold_global_id,
            "repeat":
                repeat_no,
            "fold":
                fold_no,
            "seed":
                seed,
            "train_idx":
                tr_idx,
            "valid_idx":
                va_idx,
            "ranked_features":
                rank_df[
                    "feature"
                ].tolist(),
        })

print(
    "group folds:",
    len(group_folds),
)


# =============================================================================
# 5. Group tuning
# =============================================================================

banner(
    "4) Group tuning"
)

group_rows = []
group_pred_by_config = {}

for config_no, config in enumerate(
    configs,
    start=1,
):
    pred_sum = np.zeros(
        len(df),
        dtype=float,
    )

    pred_count = np.zeros(
        len(df),
        dtype=int,
    )

    fold_pr = []

    for fold_info in group_folds:
        tr_idx = fold_info[
            "train_idx"
        ]
        va_idx = fold_info[
            "valid_idx"
        ]

        selected = fold_info[
            "ranked_features"
        ][
            : config["top_k"]
        ]

        use_cols = list(selected)

        if config["use_side"]:
            use_cols.append(
                "Side_RH"
            )

        model = make_model(
            scaler_name=
                config["scaler"],
            c_value=
                config["C"],
            weight_mode=
                config["weight_mode"],
            random_state=
                fold_info["seed"]
                + fold_info["fold"],
        )

        model.fit(
            df.iloc[
                tr_idx
            ][
                use_cols
            ].astype(float),
            y.iloc[
                tr_idx
            ],
        )

        pred = (
            model.predict_proba(
                df.iloc[
                    va_idx
                ][
                    use_cols
                ].astype(float)
            )[:, 1]
        )

        pred_sum[
            va_idx
        ] += pred

        pred_count[
            va_idx
        ] += 1

        if (
            y.iloc[
                va_idx
            ].nunique()
            == 2
        ):
            fold_pr.append(
                average_precision_score(
                    y.iloc[
                        va_idx
                    ],
                    pred,
                )
            )

    oof = np.divide(
        pred_sum,
        pred_count,
        out=np.full(
            len(df),
            np.nan,
        ),
        where=pred_count > 0,
    )

    group_pred_by_config[
        config["config_id"]
    ] = oof

    metrics = evaluate_score(
        y,
        oof,
    )

    group_rows.append({
        **config,
        **metrics,
        "fold_PR_mean": (
            float(
                np.mean(fold_pr)
            )
            if fold_pr
            else np.nan
        ),
        "fold_PR_std": (
            float(
                np.std(
                    fold_pr,
                    ddof=1,
                )
            )
            if len(fold_pr) > 1
            else np.nan
        ),
    })

    if (
        config_no % 12 == 0
        or config_no == 1
        or config_no == len(configs)
    ):
        print(
            f"[{config_no:3d}/"
            f"{len(configs)}] "
            f"{config['config_id']} | "
            f"PR={metrics['PR_AUC']:.4f} | "
            f"R80P={metrics['R80_precision']:.3f} | "
            f"R90P={metrics['R90_precision']:.3f}"
        )

group_df = pd.DataFrame(
    group_rows
)

group_df.to_csv(
    OUT_DIR
    / "group_refined_results.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 6. Purged folds
# =============================================================================

banner(
    "5) Purged folds"
)

df["time_block"] = (
    df["TimeStamp_dt"]
    .dt.floor(
        f"{TIME_BLOCK_MINUTES}min"
    )
)

unique_blocks = sorted(
    df["time_block"]
    .dropna()
    .unique()
)

purged_folds = []

purged_id = 0

for block in unique_blocks:
    block = pd.Timestamp(
        block
    )

    val_mask = (
        df["time_block"]
        == block
    )

    va_idx = np.where(
        val_mask.to_numpy()
    )[0]

    if len(va_idx) == 0:
        continue

    block_start = block
    block_end = (
        block
        + pd.Timedelta(
            minutes=
                TIME_BLOCK_MINUTES
        )
    )

    purge_start = (
        block_start
        - pd.Timedelta(
            minutes=
                PURGE_MINUTES
        )
    )

    purge_end = (
        block_end
        + pd.Timedelta(
            minutes=
                PURGE_MINUTES
        )
    )

    train_mask = ~(
        (
            df["TimeStamp_dt"]
            >= purge_start
        )
        &
        (
            df["TimeStamp_dt"]
            < purge_end
        )
    )

    val_groups = set(
        df.loc[
            val_mask,
            "sensor_group_id",
        ].tolist()
    )

    train_mask &= ~df[
        "sensor_group_id"
    ].isin(
        val_groups
    )

    tr_idx = np.where(
        train_mask.to_numpy()
    )[0]

    if (
        len(tr_idx) == 0
        or y.iloc[
            tr_idx
        ].nunique() < 2
    ):
        continue

    purged_id += 1

    rank_df = rank_features_train(
        df.iloc[
            tr_idx
        ],
        base_features,
        TARGET,
    )

    purged_folds.append({
        "purged_id":
            purged_id,
        "block":
            block,
        "train_idx":
            tr_idx,
        "valid_idx":
            va_idx,
        "ranked_features":
            rank_df[
                "feature"
            ].tolist(),
    })

print(
    "purged folds:",
    len(purged_folds),
)


# =============================================================================
# 7. Purged tuning
# =============================================================================

banner(
    "6) Purged tuning"
)

purged_rows = []
purged_pred_by_config = {}

for config_no, config in enumerate(
    configs,
    start=1,
):
    oof = np.full(
        len(df),
        np.nan,
        dtype=float,
    )

    for fold_info in purged_folds:
        tr_idx = fold_info[
            "train_idx"
        ]
        va_idx = fold_info[
            "valid_idx"
        ]

        selected = fold_info[
            "ranked_features"
        ][
            : config["top_k"]
        ]

        use_cols = list(selected)

        if config["use_side"]:
            use_cols.append(
                "Side_RH"
            )

        model = make_model(
            scaler_name=
                config["scaler"],
            c_value=
                config["C"],
            weight_mode=
                config["weight_mode"],
            random_state=
                1000
                + fold_info[
                    "purged_id"
                ],
        )

        model.fit(
            df.iloc[
                tr_idx
            ][
                use_cols
            ].astype(float),
            y.iloc[
                tr_idx
            ],
        )

        pred = (
            model.predict_proba(
                df.iloc[
                    va_idx
                ][
                    use_cols
                ].astype(float)
            )[:, 1]
        )

        oof[
            va_idx
        ] = pred

    purged_pred_by_config[
        config["config_id"]
    ] = oof

    metrics = evaluate_score(
        y,
        oof,
    )

    purged_rows.append({
        **config,
        **metrics,
    })

    if (
        config_no % 12 == 0
        or config_no == 1
        or config_no == len(configs)
    ):
        print(
            f"[{config_no:3d}/"
            f"{len(configs)}] "
            f"{config['config_id']} | "
            f"PR={metrics['PR_AUC']:.4f} | "
            f"R80P={metrics['R80_precision']:.3f} | "
            f"R90P={metrics['R90_precision']:.3f}"
        )

purged_df = pd.DataFrame(
    purged_rows
)

purged_df.to_csv(
    OUT_DIR
    / "purged_refined_results.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 8. Single model 통합
# =============================================================================

banner(
    "7) Single model 통합 순위"
)

merged = group_df.merge(
    purged_df,
    on=[
        "config_id",
        "top_k",
        "scaler",
        "C",
        "weight_mode",
        "use_side",
    ],
    suffixes=(
        "_group",
        "_purged",
    ),
)

merged["PR_min"] = merged[
    [
        "PR_AUC_group",
        "PR_AUC_purged",
    ]
].min(axis=1)

merged["PR_mean"] = merged[
    [
        "PR_AUC_group",
        "PR_AUC_purged",
    ]
].mean(axis=1)

merged["R80P_min"] = merged[
    [
        "R80_precision_group",
        "R80_precision_purged",
    ]
].min(axis=1)

merged = merged.sort_values(
    [
        "PR_min",
        "PR_mean",
        "R80P_min",
    ],
    ascending=False,
)

merged.to_csv(
    OUT_DIR
    / "single_model_combined.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    merged[
        [
            "config_id",
            "PR_AUC_group",
            "PR_AUC_purged",
            "PR_min",
            "R80_precision_group",
            "R80_recall_group",
            "R80_precision_purged",
            "R80_recall_purged",
            "R90_precision_group",
            "R90_precision_purged",
            "Top5_precision_group",
            "Top5_recall_group",
        ]
    ]
    .head(25)
    .round(4)
    .to_string(index=False)
)


# =============================================================================
# 9. Candidate pool
# =============================================================================

banner(
    "8) Ensemble candidate pool"
)

pool = []

def add_top(df_in, sort_col, n, ascending=False):
    rows = (
        df_in.sort_values(
            sort_col,
            ascending=ascending,
        )
        .head(n)
    )

    return rows[
        "config_id"
    ].tolist()

pool += add_top(
    group_df,
    "PR_AUC",
    TOP_POOL_PER_CRITERION,
)

pool += add_top(
    group_df,
    "R80_precision",
    TOP_POOL_PER_CRITERION,
)

pool += add_top(
    purged_df,
    "PR_AUC",
    TOP_POOL_PER_CRITERION,
)

pool += add_top(
    purged_df,
    "R80_precision",
    TOP_POOL_PER_CRITERION,
)

pool = list(
    dict.fromkeys(
        pool
    )
)

print(
    "pool size:",
    len(pool),
)

for x in pool:
    print(" ", x)

pd.DataFrame({
    "config_id":
        pool
}).to_csv(
    OUT_DIR
    / "ensemble_candidate_pool.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 10. Pair ensemble
# =============================================================================

banner(
    "9) Pair ensemble screening"
)

ensemble_rows = []

for a, b in combinations(
    pool,
    2,
):
    ga = group_pred_by_config[
        a
    ]
    gb = group_pred_by_config[
        b
    ]

    pa = purged_pred_by_config[
        a
    ]
    pb = purged_pred_by_config[
        b
    ]

    blend_defs = {
        "prob_mean":
            (
                (ga + gb) / 2,
                (pa + pb) / 2,
            ),

        "rank_mean":
            (
                (
                    percentile_rank_score(
                        ga
                    )
                    +
                    percentile_rank_score(
                        gb
                    )
                ) / 2,

                (
                    percentile_rank_score(
                        pa
                    )
                    +
                    percentile_rank_score(
                        pb
                    )
                ) / 2,
            ),
    }

    for blend_type, (
        group_score,
        purged_score,
    ) in blend_defs.items():

        gm = evaluate_score(
            y,
            group_score,
        )

        pm = evaluate_score(
            y,
            purged_score,
        )

        ensemble_rows.append({
            "model_a": a,
            "model_b": b,
            "blend_type": blend_type,

            "PR_AUC_group":
                gm["PR_AUC"],

            "PR_AUC_purged":
                pm["PR_AUC"],

            "PR_min":
                min(
                    gm["PR_AUC"],
                    pm["PR_AUC"],
                ),

            "R80_precision_group":
                gm[
                    "R80_precision"
                ],

            "R80_recall_group":
                gm[
                    "R80_recall"
                ],

            "R80_precision_purged":
                pm[
                    "R80_precision"
                ],

            "R80_recall_purged":
                pm[
                    "R80_recall"
                ],

            "R90_precision_group":
                gm[
                    "R90_precision"
                ],

            "R90_precision_purged":
                pm[
                    "R90_precision"
                ],

            "Top5_precision_group":
                gm[
                    "Top5_precision"
                ],

            "Top5_recall_group":
                gm[
                    "Top5_recall"
                ],
        })

ensemble_df = (
    pd.DataFrame(
        ensemble_rows
    )
)

ensemble_df = (
    ensemble_df
    .sort_values(
        [
            "PR_min",
            "PR_AUC_group",
            "R80_precision_group",
        ],
        ascending=False,
    )
)

ensemble_df.to_csv(
    OUT_DIR
    / "pair_ensemble_results.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    ensemble_df
    .head(30)
    .round(4)
    .to_string(index=False)
)


# =============================================================================
# 11. Group PR 기준 ensemble 상위
# =============================================================================

banner(
    "10) Group PR-AUC ensemble 상위"
)

group_ens_top = (
    ensemble_df
    .sort_values(
        [
            "PR_AUC_group",
            "R80_precision_group",
        ],
        ascending=False,
    )
)

print(
    group_ens_top[
        [
            "model_a",
            "model_b",
            "blend_type",
            "PR_AUC_group",
            "R80_precision_group",
            "R80_recall_group",
            "R90_precision_group",
            "Top5_precision_group",
            "Top5_recall_group",
            "PR_AUC_purged",
        ]
    ]
    .head(20)
    .round(4)
    .to_string(index=False)
)


# =============================================================================
# 12. Purged PR 기준 ensemble 상위
# =============================================================================

banner(
    "11) Purged PR-AUC ensemble 상위"
)

purged_ens_top = (
    ensemble_df
    .sort_values(
        [
            "PR_AUC_purged",
            "R80_precision_purged",
        ],
        ascending=False,
    )
)

print(
    purged_ens_top[
        [
            "model_a",
            "model_b",
            "blend_type",
            "PR_AUC_purged",
            "R80_precision_purged",
            "R80_recall_purged",
            "R90_precision_purged",
            "PR_AUC_group",
        ]
    ]
    .head(20)
    .round(4)
    .to_string(index=False)
)


# =============================================================================
# 13. 자동 요약
# =============================================================================

banner(
    "12) 자동 요약"
)

best_group_single = (
    group_df
    .sort_values(
        "PR_AUC",
        ascending=False,
    )
    .iloc[0]
)

best_purged_single = (
    purged_df
    .sort_values(
        "PR_AUC",
        ascending=False,
    )
    .iloc[0]
)

best_group_ensemble = (
    group_ens_top.iloc[0]
)

best_purged_ensemble = (
    purged_ens_top.iloc[0]
)

best_robust_ensemble = (
    ensemble_df.iloc[0]
)

lines = []

lines.append(
    f"- Single Group 최고: "
    f"{best_group_single['config_id']} | "
    f"PR={best_group_single['PR_AUC']:.4f}, "
    f"R80P={best_group_single['R80_precision']:.4f}, "
    f"R90P={best_group_single['R90_precision']:.4f}"
)

lines.append(
    f"- Single Purged 최고: "
    f"{best_purged_single['config_id']} | "
    f"PR={best_purged_single['PR_AUC']:.4f}, "
    f"R80P={best_purged_single['R80_precision']:.4f}"
)

lines.append(
    f"- Ensemble Group 최고: "
    f"{best_group_ensemble['blend_type']} | "
    f"{best_group_ensemble['model_a']} + "
    f"{best_group_ensemble['model_b']} | "
    f"PR={best_group_ensemble['PR_AUC_group']:.4f}, "
    f"R80P={best_group_ensemble['R80_precision_group']:.4f}"
)

lines.append(
    f"- Ensemble Purged 최고: "
    f"{best_purged_ensemble['blend_type']} | "
    f"{best_purged_ensemble['model_a']} + "
    f"{best_purged_ensemble['model_b']} | "
    f"PR={best_purged_ensemble['PR_AUC_purged']:.4f}, "
    f"R80P={best_purged_ensemble['R80_precision_purged']:.4f}"
)

lines.append(
    f"- Ensemble PR_min 최고: "
    f"{best_robust_ensemble['blend_type']} | "
    f"{best_robust_ensemble['model_a']} + "
    f"{best_robust_ensemble['model_b']} | "
    f"Group={best_robust_ensemble['PR_AUC_group']:.4f}, "
    f"Purged={best_robust_ensemble['PR_AUC_purged']:.4f}"
)

lines.append(
    "- 주의: ensemble 후보 선택과 평가가 같은 OOF에서 이루어졌기 때문에 "
    "앙상블 점수는 개발용 후보선정 결과다."
)

for line in lines:
    print(line)

with open(
    OUT_DIR
    / "refine_ensemble_summary.txt",
    "w",
    encoding="utf-8",
) as f:
    f.write(
        "\n".join(lines)
    )


# =============================================================================
# 완료
# =============================================================================

banner(
    "완료"
)

print(
    "결과 폴더:",
    OUT_DIR,
)

print()

print(
    "핵심 파일:"
)

print(
    "  1) group_refined_results.csv"
)

print(
    "  2) purged_refined_results.csv"
)

print(
    "  3) single_model_combined.csv"
)

print(
    "  4) ensemble_candidate_pool.csv"
)

print(
    "  5) pair_ensemble_results.csv"
)

print(
    "  6) refine_ensemble_summary.txt"
)

print()

print(
    "실행 후 7) Single model 통합 순위부터 "
    "12) 자동 요약까지 보내주세요."
)



CN7 08 - Logistic 정밀 튜닝 + OOF Blend
shape=(1211, 52), defects=17/1211, features=23

1) Sensor Group
groups: 606
positive groups: 14

2) Configs
config count: 72

3) Group folds
group folds: 25

4) Group tuning
[  1/72] k10__standard__C0.1__balanced__side0 | PR=0.4831 | R80P=0.149 | R90P=0.110
[ 12/72] k10__standard__C0.3__fixed70__side1 | PR=0.5746 | R80P=0.212 | R90P=0.121
[ 24/72] k10__robust__C0.3__fixed70__side1 | PR=0.5535 | R80P=0.230 | R90P=0.129
[ 36/72] k15__standard__C0.3__fixed70__side1 | PR=0.5791 | R80P=0.187 | R90P=0.158
[ 48/72] k15__robust__C0.3__fixed70__side1 | PR=0.5417 | R80P=0.177 | R90P=0.089
[ 60/72] k23__standard__C0.3__fixed70__side1 | PR=0.5448 | R80P=0.241 | R90P=0.151
[ 72/72] k23__robust__C0.3__fixed70__side1 | PR=0.5203 | R80P=0.144 | R90P=0.071

5) Purged folds
purged folds: 63

6) Purged tuning
[  1/72] k10__standard__C0.1__balanced__side0 | PR=0.0363 | R80P=0.014 | R90P=0.014
[ 12/72] k10__standard__C0.3__fixed70__side1 | PR=0.0192 | R80P=0.014 | R90P=